#### setup

In [ ]:
import torch
import pandas as pd
from library.embedding_utils import *
from library.data_utils import *

In [ ]:
# device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(seed=0)

In [ ]:
# parameters
dataset = 'synsum'

#### data

In [ ]:
# load data
df = pd.read_csv(f"./data/datasets/{dataset}.csv", index_col=0)

#### preprocessing

optional

In [ ]:
def clean_text(text):
    """Optional minor pre-processing."""
    # remove history and physical exam section
    text = text.replace("**History**\n", "")
    text = text.replace("**Physical Examination**\n", "")

    # remove multiple whitespaces
    text = re.sub(r"[\n]{2,}", "", text)
    return text

#### embed dataset

In [ ]:
# initialise encoder and freeze weights
encoder = ModernBERT().to(device)
for param in encoder.parameters():
    param.requires_grad = False

In [ ]:
# init collector
embeddings = []

# loop over data
for idx, row in df.iterrows():
    # embed
    text = clean_text(row['TEXT'])
    rep = encoder(text)
    embeddings.append(rep)

# store
save_tensors(embeddings, './data/embeddings/synsum.pt')